# 💧 AIRA Kalkulator Ketahanan Air: Rule-Based Calculation & AI Recommendation System

Selamat datang di **Hands-on Assignment** proyek AIRA! Notebook ini membangun dan menguji pipeline awal fitur **Kalkulator Ketahanan Air**, yang membantu rumah tangga memperkirakan kebutuhan air harian, cadangan air yang tersedia, dan estimasi durasi cadangan. Berdasarkan hasil perhitungan serta kondisi sumber air yang dipilih pengguna, sistem menentukan status dan memilih rekomendasi tindakan.

Pada pipeline ini, **perhitungan dan penetapan status dilakukan secara deterministik (rule-based)**. AI/LLM hanya menjadi lapisan opsional yang memilih dan mengurutkan tindakan dari katalog rekomendasi yang sudah ditetapkan, lalu menjelaskannya dengan bahasa yang mudah dipahami. AI tidak menghitung ulang angka, tidak mengubah status, dan tidak menjadi satu-satunya sumber rekomendasi.

### Alur Kerja (Workflow)
1. **Input Data Rumah Tangga**: lokasi opsional, jumlah anggota keluarga, sumber air utama, kondisi sumber air, serta cadangan air melalui volume aktual atau kapasitas tangki dan persentase isi. Kebutuhan MHM dapat dicatat secara opsional sebagai konteks rekomendasi.
2. **Validasi Input**: memeriksa kelengkapan, tipe data, rentang nilai, dan konsistensi data.
3. **Rule-Based Calculation**: menghitung kebutuhan air harian, cadangan air, dan estimasi durasi ketahanan.
4. **Klasifikasi Kondisi Air**: menetapkan status berdasarkan durasi dan kondisi sumber air, dengan aturan yang eksplisit.
5. **Recommendation Selection**: memilih tindakan dari katalog rekomendasi yang ditetapkan berdasarkan status dan kondisi sumber air.
6. **AI Recommendation Layer (Opsional)**: LLM memilih dan mengurutkan tindakan dari katalog yang ditetapkan serta memberi alasan singkat, tanpa menambahkan tindakan baru atau mengubah angka/status. Panggilan AI dilindungi retry, model cadangan, dan fallback: jika AI tetap gagal, rekomendasi rule-based tetap ditampilkan.
7. **Pengujian & Interpretasi**: menjalankan beberapa skenario, memeriksa batas status, dan mencatat keterbatasan.

> **Batasan penting:** angka 15 liter/orang/hari dan ambang status pada notebook ini adalah asumsi awal prototype untuk simulasi kebutuhan dasar, bukan standar kebutuhan normal semua rumah tangga atau standar resmi penetapan kekeringan. Hasil tidak membuktikan kualitas air aman diminum.

## 🎯 1. Problem Statement & Feature Description

### Latar belakang dan problem statement

Air adalah kebutuhan dasar untuk minum, memasak, kebersihan diri, dan sanitasi rumah tangga. Di wilayah rawan kekeringan, akses air sering terbatas, terutama saat kemarau berkepanjangan. Kondisi ini menambah beban perempuan, yang umumnya mengatur kebutuhan air dan kebersihan keluarga serta mencari sumber alternatif ketika persediaan menipis. Di sisi lain, informasi tentang kebutuhan air, pengelolaan cadangan, sanitasi, dan lokasi sumber air belum tentu mudah ditemukan dalam satu tempat.

**Masalah utamanya:** pengguna kesulitan memperkirakan **kebutuhan air harian dan berapa lama cadangan air dapat memenuhi kebutuhan keluarga**, serta membutuhkan **panduan tindakan yang perlu diprioritaskan** ketika persediaan mulai menipis. Pengguna juga membutuhkan informasi yang sederhana dan sesuai dengan kondisi rumah tangganya agar dapat mengambil keputusan dengan tepat.

**Solusi yang diuji pada notebook ini:** Kalkulator Ketahanan Air, salah satu fitur utama AIRA. Fitur ini menghitung kebutuhan, cadangan, dan durasi ketahanan air rumah tangga, menetapkan status kondisi air, lalu memberikan rekomendasi tindakan yang mudah dipahami.

**Pengguna sasaran:** perempuan dan keluarga, khususnya ibu rumah tangga di wilayah yang rentan kekeringan atau memiliki keterbatasan akses air. Kalkulator dapat dipakai langsung tanpa login.

### Fitur yang diuji
**Kalkulator Ketahanan Air** menghitung kebutuhan air harian, cadangan, dan estimasi durasi, lalu menetapkan **status kondisi air** dan menampilkan **rekomendasi tindakan** yang sesuai.

Alurnya: **input rumah tangga → hasil kalkulasi → status kondisi → rekomendasi → akses lanjutan ke Navigation Air atau AIRA Assistant.**

Sebagai gambaran: keluarga beranggotakan 4 orang membutuhkan sekitar 4 × 15 = 60 liter per hari. Jika cadangannya 600 liter, air diperkirakan bertahan 600 ÷ 60 = 10 hari, sehingga statusnya **Aman** (dengan asumsi tanpa kebutuhan MHM dan sumber air normal).

### Input: apa yang perlu diisi pengguna?
| Data | Wajib / Opsional | Pada pipeline ini |
|---|---|---|
| Lokasi atau wilayah | Opsional | Boleh dikosongkan; tidak dikirim ke AI |
| Jumlah anggota keluarga | Wajib | Bilangan bulat lebih dari 0 |
| Sumber air utama | Wajib | Wajib diisi (mis. sumur, PDAM, air hujan) |
| Kondisi sumber air | Wajib | Normal, Terbatas, Tidak berfungsi, atau Belum diketahui |
| Volume cadangan air | Wajib untuk menghitung durasi | Volume aktual (liter), atau kapasitas tangki × persentase isi (0–100%) |
| Kapasitas penampungan | Jika memakai penampungan | Wajib bila volume dihitung dari persentase; bila volume aktual diisi, tidak boleh melebihi kapasitas |
| Kebutuhan air harian | Ditetapkan sistem | 15 liter/orang/hari sebagai asumsi awal prototype |
| Kebutuhan MHM | Opsional | `KEBUTUHAN_MHM` dan `JUMLAH_ANGGOTA_MHM` (tidak boleh melebihi jumlah anggota keluarga) |
| Luas atap dan curah hujan | Opsional | Belum dimasukkan; panen air hujan adalah pengembangan terpisah |

### Bagaimana hasilnya dihitung?
- **Kebutuhan harian** = jumlah anggota keluarga × kebutuhan air per orang per hari.
- **Cadangan air** = volume aktual, atau kapasitas tangki × (persentase isi ÷ 100).
- **Durasi cadangan** = cadangan ÷ kebutuhan harian. Jika kebutuhan harian nol atau tidak valid, pembagian tidak dilakukan. Status ditentukan dari angka durasi yang belum dibulatkan, agar tidak bergeser karena pembulatan tampilan.

Dari durasi itulah status ditetapkan:

| Status | Ambang durasi | Aksi utama untuk pengguna |
|---|---|---|
| Aman / Relatif Mencukupi | Lebih dari 3 hari | Tanya AIRA Assistant |
| Waspada | Lebih dari 1 hari hingga 3 hari (tepat 1 dan 3 hari termasuk Waspada) | Tanya AIRA Assistant |
| Mode Krisis | Kurang dari 1 hari | Cari Titik Air di Navigation Air |
| Belum Dapat Ditentukan | Data tidak lengkap, tidak valid, atau belum mencukupi | Periksa Kembali Data |

Jika data belum cukup, sistem tidak langsung menyimpulkan Aman. Pengguna diminta melengkapi atau memperbaiki datanya terlebih dahulu.

### Keputusan tim di luar rumus dasar rancangan produk
Ada dua aturan pada pipeline ini yang melampaui rumus dasar dan perlu dievaluasi lebih lanjut:
- **Tambahan kebutuhan MHM.** Rumus dasar hanya 15 liter × jumlah anggota, sedangkan perhitungan berbasis kategori penggunaan (termasuk MHM) dirancang untuk dikembangkan kemudian. Agar kebutuhan menstruasi tidak terabaikan, pipeline menambahkan **5 liter/orang/hari** untuk setiap anggota yang memerlukan MHM. Angka ini adalah alokasi operasional prototype, bukan standar universal.
- **Kondisi sumber air.** Ambang status pada dasarnya berbasis durasi. Pipeline menambahkan penyesuaian yang berhati-hati: *Tidak berfungsi* → Mode Krisis; *Terbatas* dengan durasi lebih dari 3 hari → Waspada; *Belum diketahui* → Belum Dapat Ditentukan. Aturan ini masih keputusan prototype yang perlu divalidasi.

### Output: apa yang didapat pengguna?
Pengguna memperoleh gambaran lengkap: kebutuhan air harian, perkiraan volume cadangan, estimasi durasi, status kondisi air, **alasan** di balik status tersebut, rekomendasi tindakan, dan waktu perhitungan. Semuanya disampaikan sebagai **estimasi berdasarkan data yang dimasukkan**, bukan pengukuran langsung.

### Di mana peran AI, dan apa batasannya?
- **Angka dan status dihitung oleh kode** dengan rumus dan ambang yang konsisten serta dapat diuji.
- **AI (Gemini) membantu menyusun rekomendasi**: memilih dan mengurutkan tindakan dari katalog yang sudah disiapkan, lalu memberi alasan singkat untuk tiap pilihan.
- **AI tidak boleh** mengubah hasil perhitungan, mengarang angka, atau menentukan dan mengubah status, termasuk Mode Krisis.
- **Jika AI tidak tersedia**, hasil perhitungan dan rekomendasi rule-based tetap ditampilkan, sehingga pengguna tidak kehilangan hasil.
- **Hasil hanyalah estimasi.** Kebutuhan 15 liter/orang/hari dan ambang status adalah asumsi awal prototype, bukan standar resmi. Volume air juga tidak menunjukkan bahwa air aman diminum.

### Peran dalam final project
Kalkulator adalah pintu masuk alur AIRA, dan statusnya menunjukkan langkah berikutnya:
- **Aman:** jaga cadangan dan gunakan air dengan bijak.
- **Waspada:** berhemat dan mulai menyiapkan sumber alternatif.
- **Mode Krisis:** prioritaskan kebutuhan esensial dan cari titik air melalui **Navigation Air**.
- **Belum Dapat Ditentukan:** lengkapi atau perbaiki data, lalu hitung ulang.

Hasil terstruktur kalkulator juga menjadi bahan bagi **AIRA Assistant** untuk menjelaskan hasil tanpa menghitung ulang. Navigation Air dan AIRA Assistant tetap dapat dibuka dari menu utama pada semua status. Penyimpanan riwayat setelah login berada di luar cakupan pipeline ini.

## 📦 2. Setup dan Konstanta Prototype

Notebook ini dapat menjalankan perhitungan, rekomendasi rule-based, dan pengujian **tanpa API key**. Integrasi Gemini bersifat opsional. Untuk mengaktifkannya, pasang `google-genai` dan simpan API key pada environment variable `GEMINI_API_KEY` atau Colab Secrets. Jangan menuliskan API key langsung di notebook atau mengunggahnya ke GitHub.

In [ ]:
!pip install -q google-genai

In [ ]:
import os
import json
import math
import time
import random
import pandas as pd
from datetime import datetime, timezone
from google import genai

# Colab Secrets bersifat opsional: di luar Colab (lokal/VS Code) notebook tetap
# bisa dijalankan dengan environment variable GEMINI_API_KEY.
try:
    from google.colab import userdata
except ImportError:
    userdata = None

# SDK Gemini bersifat opsional: tanpa SDK, pipeline memakai rule-based fallback.
try:
    from google import genai
except ImportError:
    genai = None

print("Import selesai.")

Import selesai.


In [ ]:
# Konstanta Kalkulator Air

KEBUTUHAN_PER_ORANG_LITER_HARI = 15

BATAS_KRISIS_HARI = 1
BATAS_WASPADA_HARI = 3

STATUS_AMAN = "Aman / Relatif Mencukupi"
STATUS_WASPADA = "Waspada"
STATUS_KRISIS = "Mode Krisis"
STATUS_TIDAK_DITENTUKAN = "Belum Dapat Ditentukan"

print("Setup kalkulator AIRA selesai.")

Setup kalkulator AIRA selesai.


## 🧮 3. Fungsi Validasi, Perhitungan, dan Klasifikasi

### Aturan yang digunakan

- **Kebutuhan dasar harian** = jumlah seluruh anggota keluarga × asumsi kebutuhan air per orang per hari.
- **Tambahan kebutuhan air MHM** = jumlah anggota keluarga yang membutuhkan MHM × 5 liter/orang/hari. Jika tidak ada anggota yang membutuhkan MHM, tambahan kebutuhan air MHM adalah 0 liter/hari.
- **Total kebutuhan air harian** = kebutuhan dasar harian + tambahan kebutuhan air MHM.
- **Cadangan air** = volume aktual jika diketahui; jika tidak, kapasitas tangki × persentase isi / 100.
- **Durasi** = cadangan air / total kebutuhan air harian.
- Durasi **kurang dari 1 hari** → Mode Krisis; **1 sampai dengan 3 hari** → Waspada; **lebih dari 3 hari** → Aman / Relatif Mencukupi.
- Kondisi sumber **Tidak berfungsi** menjadi pemicu Mode Krisis; kondisi **Terbatas** mencegah status Aman dan minimal menghasilkan Waspada. Jika kondisi sumber **Belum diketahui**, status keseluruhan belum dapat ditentukan.
- Data tidak valid atau wajib yang tidak lengkap menghasilkan Belum Dapat Ditentukan, bukan Aman.
- Jumlah anggota yang membutuhkan MHM tidak boleh melebihi jumlah seluruh anggota keluarga.




**Catatan acuan kebutuhan air:**

Perhitungan kebutuhan air dasar AIRA menggunakan acuan minimum **15 liter/orang/hari** berdasarkan *Sphere Handbook*, yang menetapkan jumlah tersebut sebagai alokasi minimum untuk kebutuhan minum, memasak, serta kebersihan pribadi dan rumah tangga dalam respons kemanusiaan. Angka ini merupakan acuan untuk konteks darurat, bukan batas maksimum atau standar kebutuhan ideal untuk semua kondisi.




Untuk mendukung kebutuhan Manajemen Kebersihan Menstruasi (MHM), AIRA mengalokasikan tambahan **5 liter/orang/hari bagi setiap anggota keluarga yang memerlukan MHM**. Alokasi ini mempertimbangkan kebutuhan akses air untuk membersihkan diri dan tangan serta mengelola bahan menstruasi secara higienis, sebagaimana dijelaskan dalam panduan kesehatan dan kebersihan menstruasi UNICEF. Namun, angka tambahan 5 liter/orang/hari merupakan **alokasi operasional yang digunakan AIRA**, bukan angka minimum universal yang secara khusus ditetapkan oleh Sphere atau UNICEF. Besaran ini perlu dievaluasi berdasarkan praktik MHM dan kondisi penggunaan air di wilayah sasaran.



Aturan klasifikasi kondisi sumber air dan status ketahanan air merupakan logika awal kalkulator AIRA. Aturan tersebut perlu diuji dan dievaluasi bersama pihak yang memahami kebutuhan air, sanitasi, serta konteks lokal sebelum digunakan sebagai dasar rekomendasi di kondisi nyata.




>_Daftar Referensi_:
> - https://interactive.spherestandards.org/
> (Sphere Standards — Water Supply).
> - https://www.unicef.org/documents/guidance-menstrual-health-and-hygiene
> (UNICEF — Guidance on Menstrual Health and Hygiene).

In [ ]:
def _is_int(nilai):
    """True jika nilai bilangan bulat (bool tidak dihitung)."""
    return isinstance(nilai, int) and not isinstance(nilai, bool)


def _is_angka(nilai):
    """True jika nilai angka berhingga (bool, NaN, dan infinity ditolak)."""
    return (
        isinstance(nilai, (int, float))
        and not isinstance(nilai, bool)
        and math.isfinite(nilai)
    )


def klasifikasikan_status(durasi, kondisi_sumber_air):
    """Tentukan (status_durasi, status_akhir, alasan) dari durasi dan kondisi sumber."""
    if durasi < BATAS_KRISIS_HARI:
        status_durasi = STATUS_KRISIS
        alasan = (
            "Estimasi cadangan kurang dari 1 hari total kebutuhan air "
            "harian, termasuk kebutuhan MHM jika berlaku."
        )
    elif durasi <= BATAS_WASPADA_HARI:
        status_durasi = STATUS_WASPADA
        alasan = (
            "Estimasi cadangan berada pada rentang 1 sampai 3 hari "
            "total kebutuhan air harian."
        )
    else:
        status_durasi = STATUS_AMAN
        alasan = (
            "Estimasi cadangan lebih dari 3 hari berdasarkan asumsi "
            "prototype dan total kebutuhan air harian."
        )

    # Kondisi sumber air menjadi faktor konteks tambahan
    if kondisi_sumber_air == "Belum diketahui":
        return status_durasi, STATUS_TIDAK_DITENTUKAN, (
            "Durasi cadangan dapat dihitung, tetapi kondisi sumber "
            "air utama belum diketahui."
        )

    if kondisi_sumber_air == "Tidak berfungsi":
        return status_durasi, STATUS_KRISIS, (
            "Sumber air utama dilaporkan tidak berfungsi; perlu "
            "menyiapkan sumber alternatif meskipun cadangan "
            "tersimpan masih ada."
        )

    if kondisi_sumber_air == "Terbatas" and status_durasi == STATUS_AMAN:
        return status_durasi, STATUS_WASPADA, (
            "Cadangan tersimpan diperkirakan cukup lebih dari 3 hari, "
            "tetapi sumber air utama sedang terbatas."
        )

    return status_durasi, status_durasi, alasan


def hitung_ketahanan_air(
    jumlah_anggota,
    sumber_air_utama,
    kondisi_sumber_air,
    volume_cadangan_liter=None,
    kapasitas_tangki_liter=None,
    persentase_isi=None,
    lokasi=None,
    kebutuhan_mhm=False,
    jumlah_anggota_mhm=0,
    tambahan_air_mhm_per_orang=5,
    kebutuhan_per_orang=KEBUTUHAN_PER_ORANG_LITER_HARI,
):
    """Hitung ketahanan air berdasarkan aturan deterministik prototype AIRA.

    Volume cadangan menggunakan volume aktual jika tersedia; jika tidak,
    volume dihitung dari kapasitas tangki × persentase isi.

    Kebutuhan dasar dihitung berdasarkan seluruh anggota keluarga.
    Jika kebutuhan_mhm=True, tambahan kebutuhan air dihitung berdasarkan
    jumlah anggota keluarga yang membutuhkan MHM × 5 liter/orang/hari
    secara default.

    Jumlah anggota MHM tidak boleh melebihi jumlah seluruh anggota keluarga.
    """

    errors = []

    # 1. Validasi jumlah keluarga dan kebutuhan dasar
    if not _is_int(jumlah_anggota) or jumlah_anggota <= 0:
        errors.append(
            "Jumlah anggota keluarga harus berupa bilangan bulat lebih dari 0."
        )

    if not _is_angka(kebutuhan_per_orang) or kebutuhan_per_orang <= 0:
        errors.append("Kebutuhan air per orang harus berupa angka lebih dari 0.")

    if not isinstance(sumber_air_utama, str) or not sumber_air_utama.strip():
        errors.append("Sumber air utama wajib diisi.")

    # 2. Validasi input MHM
    if not isinstance(kebutuhan_mhm, bool):
        errors.append("kebutuhan_mhm harus bernilai True atau False.")

    if not _is_int(jumlah_anggota_mhm) or jumlah_anggota_mhm < 0:
        errors.append(
            "Jumlah anggota yang membutuhkan MHM harus berupa "
            "bilangan bulat 0 atau lebih."
        )
    elif (
        _is_int(jumlah_anggota)
        and jumlah_anggota > 0
        and jumlah_anggota_mhm > jumlah_anggota
    ):
        errors.append(
            "Jumlah anggota yang membutuhkan MHM tidak boleh "
            "melebihi jumlah seluruh anggota keluarga."
        )

    if not _is_angka(tambahan_air_mhm_per_orang) or tambahan_air_mhm_per_orang < 0:
        errors.append("Tambahan air MHM per orang harus berupa angka 0 atau lebih.")

    if kebutuhan_mhm is False and jumlah_anggota_mhm != 0:
        errors.append("Jika kebutuhan_mhm=False, jumlah_anggota_mhm harus 0.")

    if kebutuhan_mhm is True and jumlah_anggota_mhm == 0:
        errors.append(
            "Jika kebutuhan_mhm=True, masukkan jumlah anggota "
            "keluarga yang membutuhkan MHM."
        )

    # 3. Validasi kondisi sumber air
    pilihan_kondisi = {"Normal", "Terbatas", "Tidak berfungsi", "Belum diketahui"}

    if kondisi_sumber_air not in pilihan_kondisi:
        errors.append(
            "Kondisi sumber air harus salah satu dari: "
            f"{', '.join(sorted(pilihan_kondisi))}."
        )

    # 4. Validasi dan tentukan metode volume cadangan
    menggunakan_volume_aktual = volume_cadangan_liter is not None
    cadangan = None

    if menggunakan_volume_aktual:
        if not _is_angka(volume_cadangan_liter) or volume_cadangan_liter < 0:
            errors.append("Volume cadangan aktual harus berupa angka 0 atau lebih.")
        else:
            cadangan = float(volume_cadangan_liter)

        if kapasitas_tangki_liter is not None:
            if not _is_angka(kapasitas_tangki_liter) or kapasitas_tangki_liter <= 0:
                errors.append("Jika diisi, kapasitas tangki harus lebih dari 0.")
            elif cadangan is not None and cadangan > kapasitas_tangki_liter:
                errors.append("Volume cadangan tidak boleh melebihi kapasitas tangki.")
    else:
        if not _is_angka(kapasitas_tangki_liter) or kapasitas_tangki_liter <= 0:
            errors.append(
                "Isi volume aktual, atau masukkan kapasitas tangki lebih dari 0."
            )

        if not _is_angka(persentase_isi) or not 0 <= persentase_isi <= 100:
            errors.append("Persentase isi tangki harus berada pada rentang 0–100%.")

        if not errors:
            cadangan = float(kapasitas_tangki_liter) * (float(persentase_isi) / 100)

    # Bagian yang sama untuk hasil valid maupun tidak valid
    identitas = {
        "lokasi": lokasi,
        "jumlah_anggota": jumlah_anggota,
        "sumber_air_utama": sumber_air_utama,
        "kondisi_sumber_air": kondisi_sumber_air,
        "kebutuhan_mhm": kebutuhan_mhm,
        "jumlah_anggota_mhm": jumlah_anggota_mhm,
    }
    penutup = {
        "metode_volume": (
            "volume aktual" if menggunakan_volume_aktual else "kapasitas × persentase"
        ),
        "waktu_perhitungan_utc": datetime.now(timezone.utc).isoformat(),
    }

    # 5. Hasil belum dapat ditentukan jika input wajib tidak valid
    if errors:
        return {
            "valid": False,
            "errors": errors,
            **identitas,
            "kebutuhan_dasar_liter": None,
            "tambahan_air_mhm_liter": None,
            "kebutuhan_harian_liter": None,
            "cadangan_air_liter": None,
            "durasi_hari": None,
            "status": STATUS_TIDAK_DITENTUKAN,
            "alasan_status": "Data wajib tidak lengkap atau tidak valid.",
            **penutup,
        }

    # 6. Hitung kebutuhan dasar, tambahan MHM, dan total kebutuhan
    kebutuhan_dasar = float(jumlah_anggota) * float(kebutuhan_per_orang)
    tambahan_mhm = (
        float(jumlah_anggota_mhm) * float(tambahan_air_mhm_per_orang)
        if kebutuhan_mhm
        else 0.0
    )
    kebutuhan_harian = kebutuhan_dasar + tambahan_mhm

    # 7. Durasi cadangan memakai total kebutuhan harian
    durasi = cadangan / kebutuhan_harian

    # 8. Klasifikasi status
    status_durasi, status, alasan = klasifikasikan_status(durasi, kondisi_sumber_air)

    # 9. Kembalikan hasil kalkulasi
    return {
        "valid": True,
        "errors": [],
        **identitas,
        "jumlah_anggota": int(jumlah_anggota),
        "jumlah_anggota_mhm": int(jumlah_anggota_mhm),
        "kebutuhan_per_orang_liter_hari": float(kebutuhan_per_orang),
        "tambahan_air_mhm_per_orang_liter_hari": float(tambahan_air_mhm_per_orang),
        "kebutuhan_dasar_liter": round(kebutuhan_dasar, 2),
        "tambahan_air_mhm_liter": round(tambahan_mhm, 2),
        "kebutuhan_harian_liter": round(kebutuhan_harian, 2),
        "cadangan_air_liter": round(cadangan, 2),
        "durasi_hari": round(durasi, 2),
        "status_durasi": status_durasi,
        "status": status,
        "alasan_status": alasan,
        **penutup,
    }


print("Fungsi kalkulator siap.")

Fungsi kalkulator siap.


## 🧾 4. Input Simulasi (disesuaikan dengan form Figma)

Ubah nilai pada sel ini untuk mencoba skenario lain. Lokasi bersifat opsional. Untuk volume cadangan, isi **salah satu** metode: `VOLUME_CADANGAN_AKTUAL_LITER`, atau `KAPASITAS_TANGKI_LITER` beserta `PERSENTASE_ISI_TANGKI`.

Kebutuhan MHM dicatat melalui dua input: `KEBUTUHAN_MHM` untuk menunjukkan apakah ada anggota keluarga yang memerlukan MHM, dan `JUMLAH_ANGGOTA_MHM` untuk mencatat jumlah anggota tersebut. Jumlah anggota yang memerlukan MHM tidak boleh melebihi jumlah seluruh anggota keluarga.

Kalkulator menghitung kebutuhan air dasar sebesar 15 liter/orang/hari dan menambahkan alokasi 5 liter/orang/hari untuk setiap anggota keluarga yang memerlukan MHM. Dengan demikian, tambahan kebutuhan MHM ikut memengaruhi estimasi lama cadangan air dapat mencukupi.

**Catatan:** Acuan 15 liter/orang/hari digunakan untuk kebutuhan air dasar dalam konteks darurat. Tambahan 5 liter/orang/hari untuk MHM merupakan alokasi operasional AIRA yang perlu dievaluasi berdasarkan panduan dan kondisi lokal, bukan standar minimum universal yang ditetapkan secara khusus oleh Sphere atau UNICEF.

In [ ]:
# Contoh simulasi — ubah sesuai skenario uji yang ingin dijalankan.
LOKASI = "Rongkop, Gunungkidul"
JUMLAH_ANGGOTA = 4
SUMBER_AIR_UTAMA = "Sumur"
KONDISI_SUMBER_AIR = "Terbatas"  # Normal / Terbatas / Tidak berfungsi / Belum diketahui

# Pilih satu metode cadangan:
VOLUME_CADANGAN_AKTUAL_LITER = None  # Contoh: 450. Isi None jika memakai kapasitas × persentase.
KAPASITAS_TANGKI_LITER = 1000
PERSENTASE_ISI_TANGKI = 45

KEBUTUHAN_MHM = True
JUMLAH_ANGGOTA_MHM = 2  # Jumlah anggota keluarga yang memerlukan MHM.

hasil_kalkulasi = hitung_ketahanan_air(
    jumlah_anggota=JUMLAH_ANGGOTA,
    sumber_air_utama=SUMBER_AIR_UTAMA,
    kondisi_sumber_air=KONDISI_SUMBER_AIR,
    volume_cadangan_liter=VOLUME_CADANGAN_AKTUAL_LITER,
    kapasitas_tangki_liter=KAPASITAS_TANGKI_LITER,
    persentase_isi=PERSENTASE_ISI_TANGKI,
    lokasi=LOKASI,
    kebutuhan_mhm=KEBUTUHAN_MHM,
    jumlah_anggota_mhm=JUMLAH_ANGGOTA_MHM,
)

print(json.dumps(hasil_kalkulasi, ensure_ascii=False, indent=2))

{
  "valid": true,
  "errors": [],
  "lokasi": "Rongkop, Gunungkidul",
  "jumlah_anggota": 4,
  "sumber_air_utama": "Sumur",
  "kondisi_sumber_air": "Terbatas",
  "kebutuhan_mhm": true,
  "jumlah_anggota_mhm": 2,
  "kebutuhan_per_orang_liter_hari": 15.0,
  "tambahan_air_mhm_per_orang_liter_hari": 5.0,
  "kebutuhan_dasar_liter": 60.0,
  "tambahan_air_mhm_liter": 10.0,
  "kebutuhan_harian_liter": 70.0,
  "cadangan_air_liter": 450.0,
  "durasi_hari": 6.43,
  "status_durasi": "Aman / Relatif Mencukupi",
  "status": "Waspada",
  "alasan_status": "Cadangan tersimpan diperkirakan cukup lebih dari 3 hari, tetapi sumber air utama sedang terbatas.",
  "metode_volume": "kapasitas × persentase",
  "waktu_perhitungan_utc": "2026-10-24T01:23:08.037733+00:00"
}


## 🧭 5. Recommendation System Berbasis Aturan

Rekomendasi dipilih oleh kode berdasarkan status hasil kalkulator dan kondisi sumber air. Ini membuat perilaku sistem konsisten serta dapat diuji. AI tidak menentukan status dan tidak menjadi sumber kebenaran angka.

- **Aman / Relatif Mencukupi:** pertahankan cadangan, pantau pemakaian, dan rencanakan pengisian ulang.
- **Waspada:** hemat air untuk penggunaan non-esensial, pantau cadangan, dan mulai mencari informasi sumber alternatif.
- **Mode Krisis:** prioritaskan kebutuhan dasar, hubungi pengelola/otoritas lokal atau sumber bantuan, dan buka Navigation Air untuk memeriksa titik alternatif. Jangan menganggap titik alternatif pasti tersedia tanpa konfirmasi.
- **Belum Dapat Ditentukan:** periksa data yang belum diketahui/invalid dan hitung ulang.

Panduan sanitasi/MHM tidak boleh mendorong pengguna mengabaikan kebersihan penting. Air yang sudah tercampur bahan kimia atau limbah tidak boleh disarankan untuk minum, memasak, atau kebersihan tubuh.

Semua teks tindakan disimpan **satu kali** di `KATALOG_TINDAKAN`. Rule-based memilih ID tindakan per status, dan AI (Bagian 6) memilih dari katalog yang sama, sehingga teks tidak ganda dan AI tidak dapat menciptakan tindakan baru.


In [ ]:
# Katalog tindakan: SUMBER TUNGGAL teks rekomendasi AIRA.
# - Rule-based memilih ID tindakan per status (REKOMENDASI_PER_STATUS).
# - AI (Bagian 6) memilih dan mengurutkan ID dari katalog yang sama.
# "hanya_status" membatasi tindakan agar hanya muncul pada status tertentu.
# {sumber} dan {jumlah_mhm} diisi dari hasil kalkulator saat katalog disusun.
KATALOG_TINDAKAN = [
    {
        "id": "gunakan_air_bijak",
        "tindakan": "Gunakan air secara bijak dan pantau perubahan cadangan setiap hari.",
        "konteks": "Ketika cadangan diperkirakan mencukupi dan kebiasaan hemat perlu dijaga.",
    },
    {
        "id": "rencana_isi_ulang",
        "tindakan": "Rencanakan pengisian ulang sebelum cadangan mendekati ambang waspada.",
        "konteks": "Ketika cadangan masih cukup tetapi pengisian ulang perlu direncanakan.",
    },
    {
        "id": "pantau_sumber",
        "tindakan": "Pantau kondisi {sumber} dan konfirmasi jika terjadi perubahan pasokan.",
        "konteks": "Ketika perubahan pasokan sumber air utama perlu dipantau.",
    },
    {
        "id": "hemat_air",
        "tindakan": (
            "Kurangi penggunaan air untuk kegiatan yang bisa ditunda, tanpa "
            "mengabaikan minum, sanitasi, dan kebersihan penting."
        ),
        "konteks": "Ketika cadangan terbatas atau penghematan air diperlukan.",
    },
    {
        "id": "pantau_cadangan",
        "tindakan": (
            "Pantau sisa cadangan secara berkala dan rencanakan pengisian "
            "ulang lebih awal."
        ),
        "konteks": (
            "Untuk memantau perubahan cadangan dan membantu perencanaan "
            "kebutuhan rumah tangga."
        ),
    },
    {
        "id": "sumber_alternatif",
        "tindakan": (
            "Mulai mencari informasi sumber alternatif dan konfirmasi "
            "ketersediaannya kepada pengelola atau warga setempat."
        ),
        "konteks": (
            "Ketika sumber utama tidak berfungsi, terbatas, atau cadangan "
            "perlu segera dipersiapkan."
        ),
    },
    {
        "id": "konfirmasi_sumber",
        "tindakan": (
            "Pastikan kondisi sumber air utama diketahui; jika belum, "
            "konfirmasi kepada pengelola atau pihak setempat."
        ),
        "konteks": (
            "Ketika kondisi sumber air belum diketahui atau informasi "
            "yang tersedia belum pasti."
        ),
    },
    {
        "id": "prioritas_kebutuhan",
        "tindakan": (
            "Prioritaskan air untuk minum, memasak, dan kebutuhan "
            "kebersihan/sanitasi esensial sesuai panduan kesehatan setempat."
        ),
        "konteks": "Terutama ketika cadangan air sangat terbatas.",
    },
    {
        "id": "hubungi_pengelola",
        "tindakan": (
            "Segera hubungi pengelola sumber air, aparat setempat, atau "
            "layanan bantuan air untuk memastikan pilihan yang tersedia."
        ),
        "konteks": "Khusus Mode Krisis.",
        "hanya_status": [STATUS_KRISIS],
    },
    {
        "id": "navigation_air",
        "tindakan": (
            "Gunakan Navigation Air untuk meninjau titik alternatif, tetapi "
            "konfirmasi status, akses publik, dan ketersediaan sebelum berangkat."
        ),
        "konteks": "Khusus Mode Krisis, untuk mencari titik air alternatif.",
        "hanya_status": [STATUS_KRISIS],
    },
    {
        "id": "periksa_data",
        "tindakan": (
            "Periksa kembali jumlah anggota keluarga dan data cadangan air "
            "yang dimasukkan."
        ),
        "konteks": "Khusus status Belum Dapat Ditentukan.",
        "hanya_status": [STATUS_TIDAK_DITENTUKAN],
    },
    {
        "id": "hitung_ulang",
        "tindakan": (
            "Setelah data diperbaiki, jalankan kalkulator kembali sebelum "
            "menggunakan rekomendasi status."
        ),
        "konteks": "Khusus status Belum Dapat Ditentukan.",
        "hanya_status": [STATUS_TIDAK_DITENTUKAN],
    },
    {
        "id": "kebersihan_mhm",
        "tindakan": (
            "Pastikan kebutuhan kebersihan menstruasi tetap terpenuhi bagi "
            "{jumlah_mhm} anggota keluarga yang memerlukan MHM; jangan "
            "mengabaikan kebersihan penting demi penghematan air."
        ),
        "konteks": "Khusus ketika terdapat anggota keluarga yang membutuhkan MHM.",
    },
]

# Pilihan rule-based (deterministik) per status.
REKOMENDASI_PER_STATUS = {
    STATUS_AMAN: ["gunakan_air_bijak", "rencana_isi_ulang", "pantau_sumber"],
    STATUS_WASPADA: ["hemat_air", "pantau_cadangan", "sumber_alternatif"],
    STATUS_KRISIS: ["prioritas_kebutuhan", "hubungi_pengelola", "navigation_air"],
    STATUS_TIDAK_DITENTUKAN: ["periksa_data", "konfirmasi_sumber", "hitung_ulang"],
}

RINGKASAN_PER_STATUS = {
    STATUS_AMAN: (
        "Cadangan diperkirakan mencukupi lebih dari 3 hari berdasarkan "
        "kebutuhan harian yang dihitung oleh kalkulator, termasuk tambahan "
        "MHM jika diinput. Tetap lakukan pemantauan."
    ),
    STATUS_WASPADA: (
        "Cadangan atau kondisi sumber air menunjukkan perlunya penghematan "
        "dan persiapan sumber alternatif. Kebutuhan MHM yang diinput telah "
        "diperhitungkan dalam estimasi kebutuhan harian."
    ),
    STATUS_KRISIS: (
        "Sistem mendeteksi kondisi yang membutuhkan tindakan segera "
        "berdasarkan durasi cadangan atau sumber utama yang tidak berfungsi."
    ),
    STATUS_TIDAK_DITENTUKAN: (
        "Status belum dapat ditentukan karena data belum cukup, tidak valid, "
        "atau kondisi sumber air belum diketahui."
    ),
}

AKSI_UTAMA_PER_STATUS = {
    STATUS_AMAN: "Tanya AIRA Assistant",
    STATUS_WASPADA: "Tanya AIRA Assistant",
    STATUS_KRISIS: "Cari Titik Air melalui Navigation Air",
    STATUS_TIDAK_DITENTUKAN: "Periksa Kembali Data",
}


def perlu_mhm(data):
    """True jika ada anggota keluarga yang memerlukan MHM (input valid)."""
    jumlah = data.get("jumlah_anggota_mhm")
    return (
        data.get("kebutuhan_mhm") is True
        and isinstance(jumlah, int)
        and not isinstance(jumlah, bool)
        and jumlah > 0
    )


def susun_katalog(data):
    """Katalog tindakan yang relevan untuk status dan konteks pengguna ini."""
    status = data.get("status", STATUS_TIDAK_DITENTUKAN)
    isian = {
        "sumber": str(data.get("sumber_air_utama") or "sumber air utama").lower(),
        "jumlah_mhm": data.get("jumlah_anggota_mhm"),
    }
    katalog = []
    for item in KATALOG_TINDAKAN:
        if "hanya_status" in item and status not in item["hanya_status"]:
            continue
        if item["id"] == "kebersihan_mhm" and not perlu_mhm(data):
            continue
        katalog.append({
            "id": item["id"],
            "tindakan": item["tindakan"].format(**isian),
            "konteks": item["konteks"],
        })
    return katalog


def tindakan_wajib(data):
    """ID tindakan yang harus ada pada rekomendasi, apa pun pilihan AI."""
    status = data.get("status", STATUS_TIDAK_DITENTUKAN)
    wajib = []
    if status == STATUS_KRISIS:
        wajib.append("navigation_air")
    if status == STATUS_TIDAK_DITENTUKAN:
        wajib.append("periksa_data")
    if perlu_mhm(data):
        wajib.append("kebersihan_mhm")
    return wajib


def pilih_rekomendasi(data):
    """Pilih rekomendasi dasar deterministik sesuai hasil kalkulator."""
    status = data.get("status", STATUS_TIDAK_DITENTUKAN)
    if not data.get("valid", False) or status not in REKOMENDASI_PER_STATUS:
        status = STATUS_TIDAK_DITENTUKAN

    id_dipilih = list(REKOMENDASI_PER_STATUS[status])
    if perlu_mhm(data):
        id_dipilih.append("kebersihan_mhm")

    katalog = {item["id"]: item for item in susun_katalog({**data, "status": status})}

    return {
        "status_sumber": status,
        "ringkasan_rule_based": RINGKASAN_PER_STATUS[status],
        "rekomendasi_id": id_dipilih,
        "rekomendasi": [katalog[item_id]["tindakan"] for item_id in id_dipilih],
        "aksi_utama": AKSI_UTAMA_PER_STATUS[status],
        "tautan_fitur_lanjutan": {
            "Navigation Air": status == STATUS_KRISIS,
            "AIRA Assistant": status in {STATUS_AMAN, STATUS_WASPADA, STATUS_KRISIS},
        },
    }


def susun_teks_fallback(rekomendasi):
    """Teks tampilan dari rekomendasi rule-based (dipakai jika AI gagal)."""
    daftar = "\n".join(
        f"{i}. {item}" for i, item in enumerate(rekomendasi["rekomendasi"], 1)
    )
    return (
        rekomendasi["ringkasan_rule_based"]
        + "\n\nLangkah yang bisa dilakukan:\n"
        + daftar
    )


rekomendasi_rule_based = pilih_rekomendasi(hasil_kalkulasi)

print("STATUS:", rekomendasi_rule_based["status_sumber"])
print("RINGKASAN:", rekomendasi_rule_based["ringkasan_rule_based"])
print("AKSI UTAMA:", rekomendasi_rule_based["aksi_utama"])

for i, action in enumerate(rekomendasi_rule_based["rekomendasi"], 1):
    print(f"{i}. {action}")

STATUS: Waspada
RINGKASAN: Cadangan atau kondisi sumber air menunjukkan perlunya penghematan dan persiapan sumber alternatif. Kebutuhan MHM yang diinput telah diperhitungkan dalam estimasi kebutuhan harian.
AKSI UTAMA: Tanya AIRA Assistant
1. Kurangi penggunaan air untuk kegiatan yang bisa ditunda, tanpa mengabaikan minum, sanitasi, dan kebersihan penting.
2. Pantau sisa cadangan secara berkala dan rencanakan pengisian ulang lebih awal.
3. Mulai mencari informasi sumber alternatif dan konfirmasi ketersediaannya kepada pengelola atau warga setempat.
4. Pastikan kebutuhan kebersihan menstruasi tetap terpenuhi bagi 2 anggota keluarga yang memerlukan MHM; jangan mengabaikan kebersihan penting demi penghematan air.


## 🤖 6. AI Recommendation Layer (Retry, Model Cadangan, dan Fallback)

Pada tahap ini AI (Gemini) **memilih dan mengurutkan 3–5 tindakan** dari katalog tindakan yang sama dengan yang dipakai rule-based (Bagian 5), lalu memberi alasan singkat untuk tiap pilihan. Python tetap menentukan angka dan status; teks tindakan selalu diambil dari katalog sehingga AI tidak dapat menambahkan tindakan baru. Hasil AI divalidasi sebelum ditampilkan: ID di luar katalog dibuang, dan tindakan wajib (misalnya MHM, serta Navigation Air pada Mode Krisis) ditambahkan jika AI melewatkannya.

Data yang dikirim ke AI mencakup kebutuhan dasar, jumlah anggota yang memerlukan MHM, tambahan kebutuhan MHM, total kebutuhan harian, cadangan, estimasi durasi, kondisi sumber, dan status. Lokasi tidak dikirim.

**Kendala yang ditemukan (error 503):** pada uji sebelumnya AI gagal dengan `503 UNAVAILABLE` ("This model is currently experiencing high demand"), padahal uji koneksi berhasil. Ini error sementara di sisi server Gemini, bukan kesalahan kode atau API key, tetapi dulu langsung membuat pipeline jatuh ke fallback. Perbaikannya: semua panggilan AI melalui satu fungsi `panggil_gemini()` yang (1) mencoba ulang error sementara (429/500/502/503/504) dengan jeda bertambah, (2) berpindah ke model cadangan, dan (3) baru memakai rule-based fallback jika semuanya gagal.

Simpan API key di environment variable `GEMINI_API_KEY` atau Colab Secrets. Jangan commit API key. Urutan model dapat diatur lewat `GEMINI_MODELS` (dipisahkan koma); default: `gemini-3.8-flash`, `gemini-3.6-flash`, `gemini-3.5-flash-lite`. Nama model dapat berubah atau dihentikan, jadi cek daftar terbaru di dokumentasi Gemini API. Kegagalan AI tidak boleh menghentikan kalkulator.


In [ ]:
# ===== Konfigurasi Gemini =====
# Urutan = prioritas: model utama dulu, lalu model cadangan jika model utama
# sedang sibuk atau tidak tersedia. Ubah lewat environment variable
# GEMINI_MODELS (dipisahkan koma) tanpa mengedit kode. Nama model bisa
# berubah/dihentikan Google, jadi cek daftar terbaru di dokumentasi Gemini API.
GEMINI_MODELS = [
    nama.strip()
    for nama in os.getenv(
        "GEMINI_MODELS", "gemini-3.8-flash,gemini-3.6-flash,gemini-3.5-flash-lite"
    ).split(",")
    if nama.strip()
]

# Error sementara di sisi server/kuota: layak dicoba ulang.
KODE_ERROR_SEMENTARA = {429, 500, 502, 503, 504}
# Error kredensial: mencoba ulang atau ganti model tidak akan membantu.
KODE_ERROR_KUNCI = {401, 403}


class GeminiTidakTersedia(RuntimeError):
    """AI tidak dapat dipakai (API key/SDK tidak ada, atau semua model gagal)."""


def ambil_api_key():
    """Ambil API key dari environment variable, lalu dari Colab Secrets."""
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key and userdata is not None:
        try:
            api_key = userdata.get("GEMINI_API_KEY")
        except Exception:  # secret belum dibuat / akses notebook belum diizinkan
            api_key = None
    if api_key:
        os.environ["GEMINI_API_KEY"] = api_key
    return api_key


_cache_client = {}


def get_client():
    """Buat client Gemini satu kali dan pakai ulang."""
    if genai is None:
        raise GeminiTidakTersedia("SDK google-genai belum terpasang")
    api_key = ambil_api_key()
    if not api_key:
        raise GeminiTidakTersedia("API key GEMINI_API_KEY belum tersedia")
    if api_key not in _cache_client:
        _cache_client[api_key] = genai.Client(api_key=api_key)
    return _cache_client[api_key]


def panggil_gemini(
    prompt,
    *,
    json_mode=True,
    temperature=0.2,
    models=None,
    max_percobaan=4,
    jeda_awal=2.0,
    client=None,
    verbose=True,
):
    """Satu-satunya pintu panggilan ke Gemini API.

    Untuk tiap model: error sementara (429/500/502/503/504) dicoba ulang dengan
    jeda yang bertambah (exponential backoff + jitter). Jika masih gagal atau
    modelnya tidak tersedia (mis. 404), lanjut ke model berikutnya.
    Mengembalikan (teks, nama_model); melempar GeminiTidakTersedia jika semua gagal.
    """
    client = client or get_client()
    models = models or GEMINI_MODELS

    config = {"temperature": temperature}
    if json_mode:
        config["response_mime_type"] = "application/json"

    catatan = []
    for model in models:
        for percobaan in range(1, max_percobaan + 1):
            try:
                response = client.models.generate_content(
                    model=model, contents=prompt, config=config
                )
                if not response.text:
                    raise ValueError("Respons AI kosong")
                return response.text, model

            except Exception as exc:
                kode = getattr(exc, "code", None)

                if kode in KODE_ERROR_KUNCI:
                    raise GeminiTidakTersedia(
                        f"API key ditolak (HTTP {kode})"
                    ) from exc

                if kode in KODE_ERROR_SEMENTARA and percobaan < max_percobaan:
                    jeda = jeda_awal * 2 ** (percobaan - 1) * random.uniform(1, 1.25)
                    if verbose:
                        print(
                            f"[Gemini] {model} sibuk (HTTP {kode}), percobaan "
                            f"{percobaan}/{max_percobaan}; ulangi dalam {jeda:.0f} dtk..."
                        )
                    time.sleep(jeda)
                    continue

                catatan.append(f"{model}: {type(exc).__name__} {kode or ''}".strip())
                if verbose:
                    print(f"[Gemini] {model} gagal ({catatan[-1]}); coba model berikutnya.")
                break

    raise GeminiTidakTersedia("Semua model gagal -> " + "; ".join(catatan))


def uji_koneksi_gemini():
    """Cek cepat apakah Gemini dapat dipakai. Tidak menghentikan notebook jika gagal."""
    try:
        teks, model = panggil_gemini(
            "Balas dengan teks: Koneksi Gemini berhasil.",
            json_mode=False,
            max_percobaan=2,
        )
        print(f"Koneksi Gemini berhasil (model: {model}).")
        print("Respons:", teks.strip())
        return True
    except GeminiTidakTersedia as exc:
        print("Gemini belum dapat dipakai:", exc)
        print("Pipeline tetap berjalan dengan rule-based fallback.")
        return False


uji_koneksi_gemini()

[Gemini] gemini-3.8-flash sibuk (HTTP 503), percobaan 1/2; ulangi dalam 2 dtk...
[Gemini] gemini-3.8-flash gagal (gemini-3.8-flash: ServerError 503); coba model berikutnya.
Koneksi Gemini berhasil (model: gemini-3.6-flash).
Respons: Koneksi Gemini berhasil.


True

In [ ]:
BATAS_REKOMENDASI_AI = 5

# Data hasil kalkulator yang dikirim ke AI (lokasi sengaja tidak dikirim).
KUNCI_KONTEKS_AI = [
    "status", "alasan_status", "durasi_hari", "cadangan_air_liter",
    "jumlah_anggota", "kondisi_sumber_air", "kebutuhan_mhm",
    "jumlah_anggota_mhm", "kebutuhan_dasar_liter", "tambahan_air_mhm_liter",
    "kebutuhan_harian_liter",
]


def bangun_prompt_ai(data, rekomendasi, katalog):
    konteks = {kunci: data.get(kunci) for kunci in KUNCI_KONTEKS_AI}
    return f"""
Anda adalah komponen AI recommendation system untuk AIRA,
sebuah prototipe pendamping ketahanan air rumah tangga.

TUGAS:
Pilih dan urutkan 3 sampai 5 tindakan yang paling relevan
berdasarkan konteks pengguna. Jangan sekadar menulis ulang
semua rekomendasi yang tersedia.

DATA KALKULATOR:
{json.dumps(konteks, ensure_ascii=False)}

RINGKASAN RULE-BASED:
{rekomendasi.get("ringkasan_rule_based", "")}

REKOMENDASI RULE-BASED YANG SUDAH TERSEDIA:
{json.dumps(rekomendasi.get("rekomendasi", []), ensure_ascii=False)}

KATALOG TINDAKAN YANG DIIZINKAN:
{json.dumps(katalog, ensure_ascii=False)}

ATURAN PEMILIHAN:
1. Gunakan status, kondisi sumber air, dan durasi cadangan
   untuk menentukan prioritas tindakan.
2. Pilih hanya ID yang tersedia dalam katalog tindakan.
3. Urutkan ID dari prioritas tertinggi ke terendah.
4. Jika kebutuhan_mhm bernilai true dan jumlah_anggota_mhm
   lebih dari 0, sertakan tindakan "kebersihan_mhm".
5. Jika kondisi sumber air "Belum diketahui", prioritaskan
   konfirmasi kondisi sumber air. Jangan menganggap sumber
   air pasti tersedia atau tidak berfungsi.
6. Jika kondisi sumber air "Tidak berfungsi", prioritaskan
   persiapan sumber alternatif.
7. Jika status "Mode Krisis", sertakan tindakan "navigation_air".
8. Jika status "Belum Dapat Ditentukan", sertakan "periksa_data",
   jelaskan keterbatasan informasi dan jangan memberikan
   kepastian yang tidak didukung data.
9. Jangan mengubah, menghitung ulang, atau menyimpulkan status
   kalkulator yang berbeda dari hasil Python.
10. Jangan membuat angka, lokasi, titik air, jam buka, biaya,
    atau klaim kualitas air yang tidak tersedia dalam data.
11. Jangan menyatakan air aman diminum hanya berdasarkan hasil
    kalkulator ketahanan air.
12. Berikan alasan singkat untuk setiap pilihan yang mengacu
    pada konteks pengguna.
13. Gunakan Bahasa Indonesia yang sederhana, empatik,
    dan tidak menghakimi.

KELUARAN:
Kembalikan JSON valid saja dengan struktur:
{{
  "ringkasan": "penjelasan singkat kondisi pengguna",
  "selected_ids": ["id_prioritas_1", "id_prioritas_2"],
  "alasan": {{
    "id_prioritas_1": "alasan singkat",
    "id_prioritas_2": "alasan singkat"
  }}
}}

Jangan memasukkan ID yang tidak ada dalam katalog.
"""


def parse_json_ai(teks):
    """Parse JSON dari AI; toleran terhadap pembungkus ```json ... ```."""
    teks = teks.strip()
    if teks.startswith("```"):
        teks = teks.strip("`").strip()
        if teks.lower().startswith("json"):
            teks = teks[4:]
    hasil = json.loads(teks)
    if not isinstance(hasil, dict):
        raise ValueError("Format JSON AI tidak valid")
    return hasil


def pilih_id_final(id_dari_ai, katalog_by_id, wajib, batas=BATAS_REKOMENDASI_AI):
    """Validasi pilihan AI: buang ID asing/duplikat, tambahkan tindakan wajib,
    lalu batasi jumlahnya tanpa pernah membuang tindakan wajib."""
    if not isinstance(id_dari_ai, list):
        raise ValueError("Format selected_ids tidak valid")

    pilihan = list(dict.fromkeys(
        item for item in id_dari_ai
        if isinstance(item, str) and item in katalog_by_id
    ))
    if not pilihan:
        raise ValueError("AI tidak memilih rekomendasi yang valid")

    for item in wajib:
        if item not in pilihan:
            pilihan.append(item)

    for item in reversed(list(pilihan)):
        if len(pilihan) <= batas:
            break
        if item not in wajib:
            pilihan.remove(item)

    return pilihan


def generate_ai_recommendation(data, rekomendasi):
    """AI recommendation system untuk memilih dan memprioritaskan tindakan AIRA.

    Python tetap menentukan hasil kalkulasi dan status. Gemini memilih dan
    mengurutkan tindakan dari katalog yang diizinkan serta memberi alasan.
    Teks tindakan selalu berasal dari katalog (AI tidak dapat membuat tindakan
    baru). Jika AI tidak tersedia atau keluarannya tidak valid, sistem memakai
    rekomendasi rule-based sebagai fallback.
    """
    fallback = susun_teks_fallback(rekomendasi)

    try:
        katalog = susun_katalog(data)
        katalog_by_id = {item["id"]: item for item in katalog}

        teks_json, model = panggil_gemini(bangun_prompt_ai(data, rekomendasi, katalog))
        hasil_ai = parse_json_ai(teks_json)

        selected_ids = pilih_id_final(
            hasil_ai.get("selected_ids", []), katalog_by_id, tindakan_wajib(data)
        )

        alasan_ai = hasil_ai.get("alasan")
        if not isinstance(alasan_ai, dict):
            alasan_ai = {}

        ringkasan_ai = hasil_ai.get("ringkasan")
        if not isinstance(ringkasan_ai, str) or not ringkasan_ai.strip():
            ringkasan_ai = rekomendasi.get("ringkasan_rule_based", "")

        baris = []
        for nomor, item_id in enumerate(selected_ids, start=1):
            teks_item = f"{nomor}. {katalog_by_id[item_id]['tindakan']}"
            alasan = alasan_ai.get(item_id, "")
            if isinstance(alasan, str) and alasan.strip():
                teks_item += f"\n   Alasan: {alasan.strip()}"
            baris.append(teks_item)

        return {
            "mode": f"AI recommendation ({model})",
            "teks": (
                ringkasan_ai.strip()
                + "\n\nRekomendasi prioritas AIRA:\n"
                + "\n".join(baris)
            ),
            "rekomendasi_ai": selected_ids,
        }

    except Exception as exc:
        return {
            "mode": f"rule-based fallback (AI gagal: {type(exc).__name__}: {exc})",
            "teks": fallback,
            "rekomendasi_ai": [],
        }


# Jalankan AI recommendation system
hasil_penyampaian = generate_ai_recommendation(hasil_kalkulasi, rekomendasi_rule_based)

print("MODE:", hasil_penyampaian["mode"])
print("\n" + hasil_penyampaian["teks"])

[Gemini] gemini-3.8-flash sibuk (HTTP 503), percobaan 1/4; ulangi dalam 2 dtk...
[Gemini] gemini-3.8-flash sibuk (HTTP 503), percobaan 2/4; ulangi dalam 5 dtk...
[Gemini] gemini-3.8-flash sibuk (HTTP 503), percobaan 3/4; ulangi dalam 8 dtk...
MODE: AI recommendation (gemini-3.8-flash)

Kondisi berada dalam status Waspada karena sumber air utama terbatas, meskipun cadangan tersimpan diperkirakan masih mencukupi untuk sekitar 6 hari.

Rekomendasi prioritas AIRA:
1. Kurangi penggunaan air untuk kegiatan yang bisa ditunda, tanpa mengabaikan minum, sanitasi, dan kebersihan penting.
   Alasan: Penghematan pada kegiatan non-esensial perlu dilakukan untuk menjaga ketahanan cadangan selama pasokan sumber utama terbatas.
2. Pastikan kebutuhan kebersihan menstruasi tetap terpenuhi bagi 2 anggota keluarga yang memerlukan MHM; jangan mengabaikan kebersihan penting demi penghematan air.
   Alasan: Menjamin 2 anggota keluarga tetap terpenuhi kebutuhan kebersihan menstruasinya tanpa dikorbankan demi p

## 🧪 7. Pengujian Pipeline

Pengujian berikut memeriksa batas klasifikasi (kurang dari 1 hari, tepat 1 hari, tepat 3 hari, dan lebih dari 3 hari), kondisi sumber yang tidak berfungsi/terbatas/belum diketahui, input tidak valid, serta perhitungan tambahan kebutuhan MHM. Lapisan AI diuji pada sel terpisah di bawahnya tanpa memanggil API (respons disimulasikan): retry, model cadangan, validasi ID, tindakan wajib, dan fallback.

Pengujian MHM memastikan bahwa kebutuhan dasar dan tambahan MHM dijumlahkan sebelum durasi cadangan dihitung, serta jumlah anggota yang memerlukan MHM tidak melebihi jumlah seluruh anggota keluarga. Ini merupakan bukti awal bahwa fungsi mengikuti aturan yang dinyatakan, bukan bukti bahwa angka kebutuhan maupun ambang status sudah tervalidasi untuk semua kondisi nyata.

In [ ]:
def skenario_durasi(durasi, kondisi="Normal"):
    # 1 orang × 15 L/hari; volume cadangan disusun agar durasinya sesuai target.
    return hitung_ketahanan_air(
        jumlah_anggota=1,
        sumber_air_utama="Simulasi",
        kondisi_sumber_air=kondisi,
        volume_cadangan_liter=durasi * KEBUTUHAN_PER_ORANG_LITER_HARI,
    )


# Skenario perhitungan MHM:
# 4 anggota, 2 memerlukan MHM, cadangan 450 liter.
hasil_uji_mhm = hitung_ketahanan_air(
    jumlah_anggota=4,
    sumber_air_utama="Sumur",
    kondisi_sumber_air="Normal",
    volume_cadangan_liter=450,
    kebutuhan_mhm=True,
    jumlah_anggota_mhm=2,
)

# Skenario input tidak valid: jumlah anggota MHM melebihi jumlah keluarga.
hasil_uji_jumlah_mhm_tidak_valid = hitung_ketahanan_air(
    jumlah_anggota=2,
    sumber_air_utama="Sumur",
    kondisi_sumber_air="Normal",
    volume_cadangan_liter=100,
    kebutuhan_mhm=True,
    jumlah_anggota_mhm=3,
)


test_cases = [
    ("0.5 hari → Krisis", skenario_durasi(0.5), STATUS_KRISIS),
    ("1 hari → Waspada", skenario_durasi(1), STATUS_WASPADA),
    ("3 hari → Waspada", skenario_durasi(3), STATUS_WASPADA),
    ("3.01 hari → Aman", skenario_durasi(3.01), STATUS_AMAN),
    (
        "Sumber tidak berfungsi → Krisis",
        skenario_durasi(5, "Tidak berfungsi"),
        STATUS_KRISIS,
    ),
    (
        "Sumber terbatas membatasi status aman",
        skenario_durasi(5, "Terbatas"),
        STATUS_WASPADA,
    ),
    (
        "Kondisi sumber belum diketahui",
        skenario_durasi(5, "Belum diketahui"),
        STATUS_TIDAK_DITENTUKAN,
    ),
    (
        "Anggota nol → tidak ditentukan",
        hitung_ketahanan_air(
            0, "Sumur", "Normal", volume_cadangan_liter=100
        ),
        STATUS_TIDAK_DITENTUKAN,
    ),
    (
        "Persentase >100 → tidak ditentukan",
        hitung_ketahanan_air(
            2, "Sumur", "Normal",
            kapasitas_tangki_liter=500,
            persentase_isi=120,
        ),
        STATUS_TIDAK_DITENTUKAN,
    ),
    (
        "MHM dihitung dalam kebutuhan harian",
        hasil_uji_mhm,
        STATUS_AMAN,
    ),
    (
        "Jumlah anggota MHM melebihi jumlah keluarga",
        hasil_uji_jumlah_mhm_tidak_valid,
        STATUS_TIDAK_DITENTUKAN,
    ),
]

rows = []

for nama, hasil, expected in test_cases:
    actual = hasil["status"]
    rows.append({
        "Skenario": nama,
        "Status aktual": actual,
        "Status yang diharapkan": expected,
        "Lulus": actual == expected,
    })

test_results = pd.DataFrame(rows)
display(test_results)

assert test_results["Lulus"].all(), (
    "Ada pengujian yang gagal; periksa aturan klasifikasi/validasi."
)

# Periksa nilai perhitungan MHM secara terpisah.
assert hasil_uji_mhm["valid"] is True
assert abs(hasil_uji_mhm["kebutuhan_dasar_liter"] - 60) < 1e-9
assert abs(hasil_uji_mhm["tambahan_air_mhm_liter"] - 10) < 1e-9
assert abs(hasil_uji_mhm["kebutuhan_harian_liter"] - 70) < 1e-9
assert hasil_uji_mhm["durasi_hari"] == round(450 / 70, 2)

assert hasil_uji_jumlah_mhm_tidak_valid["valid"] is False

print(
    f"Semua {len(test_results)} pengujian status lulus, "
    "dan pengujian nilai perhitungan MHM lulus."
)

,Skenario,Status aktual,Status yang diharapkan,Lulus
0,0.5 hari → Krisis,Mode Krisis,Mode Krisis,True
1,1 hari → Waspada,Waspada,Waspada,True
2,3 hari → Waspada,Waspada,Waspada,True
3,3.01 hari → Aman,Aman / Relatif Mencukupi,Aman / Relatif Mencukupi,True
4,Sumber tidak berfungsi → Krisis,Mode Krisis,Mode Krisis,True
5,Sumber terbatas membatasi status aman,Waspada,Waspada,True
6,Kondisi sumber belum diketahui,Belum Dapat Ditentukan,Belum Dapat Ditentukan,True
7,Anggota nol → tidak ditentukan,Belum Dapat Ditentukan,Belum Dapat Ditentukan,True
8,Persentase >100 → tidak ditentukan,Belum Dapat Ditentukan,Belum Dapat Ditentukan,True
9,MHM dihitung dalam kebutuhan harian,Aman / Relatif Mencukupi,Aman / Relatif Mencukupi,True


Semua 11 pengujian status lulus, dan pengujian nilai perhitungan MHM lulus.


In [ ]:
# --- Pengujian lapisan AI TANPA memanggil API (respons disimulasikan) ---

class _ErrorAPI(Exception):
    """Meniru error Gemini API yang membawa kode HTTP."""
    def __init__(self, code):
        super().__init__(f"HTTP {code}")
        self.code = code


class _ClientPalsu:
    """Client palsu: tiap panggilan mengembalikan/melempar item berikutnya."""
    def __init__(self, urutan):
        self.urutan = list(urutan)
        self.dipanggil = []
        self.models = self

    def generate_content(self, model, contents, config=None):
        self.dipanggil.append(model)
        aksi = self.urutan.pop(0)
        if isinstance(aksi, Exception):
            raise aksi
        return type("Respons", (), {"text": aksi})()


def _uji_panggil(urutan, **opsi):
    client = _ClientPalsu(urutan)
    try:
        hasil = panggil_gemini(
            "x", models=["utama", "cadangan"], client=client,
            jeda_awal=0, verbose=False, **opsi,
        )
    except GeminiTidakTersedia as exc:
        hasil = exc
    return hasil, client.dipanggil


def _uji_ai(data, respons):
    """Jalankan generate_ai_recommendation dengan respons Gemini palsu."""
    asli = globals()["panggil_gemini"]

    def palsu(prompt, **kwargs):
        if isinstance(respons, Exception):
            raise respons
        return respons, "model-simulasi"

    globals()["panggil_gemini"] = palsu
    try:
        return generate_ai_recommendation(data, pilih_rekomendasi(data))
    finally:
        globals()["panggil_gemini"] = asli


data_waspada_mhm = hitung_ketahanan_air(
    4, "Sumur", "Terbatas", volume_cadangan_liter=450,
    kebutuhan_mhm=True, jumlah_anggota_mhm=2,
)
data_krisis = skenario_durasi(0.5)

# Retry & model cadangan
h1, d1 = _uji_panggil([_ErrorAPI(503), _ErrorAPI(503), "{}"])
h2, d2 = _uji_panggil([_ErrorAPI(503), _ErrorAPI(503), "{}"], max_percobaan=2)
h3, d3 = _uji_panggil([_ErrorAPI(404), "{}"])
h4, d4 = _uji_panggil([_ErrorAPI(403)])
h5, d5 = _uji_panggil([_ErrorAPI(503)] * 2, max_percobaan=1)

# Guardrail hasil AI
ai_a = _uji_ai(data_waspada_mhm, json.dumps({
    "ringkasan": "Ringkasan AI.",
    "selected_ids": ["hemat_air", "id_palsu", "pantau_cadangan", "sumber_alternatif",
                     "prioritas_kebutuhan", "konfirmasi_sumber"],
    "alasan": {"hemat_air": "Sumber sedang terbatas."},
}))
ai_b = _uji_ai(data_krisis, json.dumps({
    "ringkasan": "Ringkasan AI.",
    "selected_ids": ["periksa_data", "prioritas_kebutuhan"],
}))
ai_c = _uji_ai(data_krisis, _ErrorAPI(503))
ai_d = _uji_ai(data_krisis, "ini bukan JSON")

uji_ai = [
    ("503 dicoba ulang pada model yang sama", h1 == ("{}", "utama") and d1 == ["utama"] * 3),
    ("503 berulang → pindah ke model cadangan", h2 == ("{}", "cadangan") and d2 == ["utama", "utama", "cadangan"]),
    ("404 (model tidak ada) → langsung model cadangan", h3 == ("{}", "cadangan") and d3 == ["utama", "cadangan"]),
    ("403 (API key ditolak) → berhenti tanpa retry", isinstance(h4, GeminiTidakTersedia) and d4 == ["utama"]),
    ("Semua model gagal → GeminiTidakTersedia", isinstance(h5, GeminiTidakTersedia)),
    ("ID di luar katalog dibuang & maksimal 5 tindakan",
     "id_palsu" not in ai_a["rekomendasi_ai"] and len(ai_a["rekomendasi_ai"]) == 5),
    ("Tindakan MHM wajib tetap ada walau AI melewatkannya", "kebersihan_mhm" in ai_a["rekomendasi_ai"]),
    ("Mode Krisis: navigation_air ditambahkan, periksa_data ditolak",
     ai_b["rekomendasi_ai"] == ["prioritas_kebutuhan", "navigation_air"]),
    ("Respons AI berupa error → rule-based fallback",
     ai_c["mode"].startswith("rule-based fallback") and ai_c["rekomendasi_ai"] == []),
    ("Respons AI bukan JSON → rule-based fallback", ai_d["mode"].startswith("rule-based fallback")),
]

hasil_uji_ai = pd.DataFrame(uji_ai, columns=["Skenario lapisan AI", "Lulus"])
display(hasil_uji_ai)
assert hasil_uji_ai["Lulus"].all(), "Ada pengujian lapisan AI yang gagal."
print(f"Semua {len(hasil_uji_ai)} pengujian lapisan AI lulus (tanpa memanggil API).")

,Skenario lapisan AI,Lulus
0,503 dicoba ulang pada model yang sama,True
1,503 berulang → pindah ke model cadangan,True
2,404 (model tidak ada) → langsung model cadangan,True
3,403 (API key ditolak) → berhenti tanpa retry,True
4,Semua model gagal → GeminiTidakTersedia,True
5,ID di luar katalog dibuang & maksimal 5 tindakan,True
6,Tindakan MHM wajib tetap ada walau AI melewatk...,True
7,"Mode Krisis: navigation_air ditambahkan, perik...",True
8,Respons AI berupa error → rule-based fallback,True
9,Respons AI bukan JSON → rule-based fallback,True


Semua 10 pengujian lapisan AI lulus (tanpa memanggil API).


## 📊 8. Interpretasi Hasil Simulasi

Baca hasil skenario utama di atas dengan membedakan tiga hal:

1. **Angka hasil kalkulasi** dihitung oleh fungsi Python. Kebutuhan air dasar menggunakan acuan 15 liter/orang/hari untuk konteks darurat. Jika ada anggota keluarga yang memerlukan MHM, tambahan 5 liter/orang/hari per anggota tersebut ikut dihitung sesuai alokasi operasional AIRA.
2. **Status** dipilih oleh aturan sistem berdasarkan durasi dan kondisi sumber air. Contohnya, cadangan dapat diperkirakan cukup lebih dari 3 hari tetapi status menjadi Waspada bila sumber utama sedang terbatas.
3. **Rekomendasi** dipilih dari satu katalog tindakan. Rule-based memilih secara deterministik; AI, jika aktif, hanya memilih dan mengurutkan tindakan dari katalog yang sama beserta alasannya; jika AI gagal, teks rule-based tetap tersedia.

Hasil adalah estimasi berdasarkan input, bukan pengukuran sensor real-time. Kondisi sumber air dan cadangan dapat berubah. Kalkulator tidak menentukan kualitas atau keamanan air minum. Angka tambahan MHM merupakan alokasi operasional AIRA dan tidak boleh dianggap sebagai standar minimum universal yang telah ditetapkan secara khusus oleh Sphere atau UNICEF.

In [ ]:

# Ringkasan terstruktur yang mudah disimpan atau ditampilkan sebagai output awal.
ringkasan_output = {
    "input": {
        "lokasi": hasil_kalkulasi.get("lokasi"),
        "jumlah_anggota": hasil_kalkulasi.get("jumlah_anggota"),
        "sumber_air_utama": hasil_kalkulasi.get("sumber_air_utama"),
        "kondisi_sumber_air": hasil_kalkulasi.get("kondisi_sumber_air"),
        "metode_volume": hasil_kalkulasi.get("metode_volume"),
        "kebutuhan_mhm": hasil_kalkulasi.get("kebutuhan_mhm"),
        "jumlah_anggota_mhm": hasil_kalkulasi.get("jumlah_anggota_mhm"),
    },
    "hasil_kalkulasi": {
        "kebutuhan_dasar_liter": hasil_kalkulasi.get("kebutuhan_dasar_liter"),
        "tambahan_air_mhm_liter": hasil_kalkulasi.get("tambahan_air_mhm_liter"),
        "kebutuhan_harian_liter": hasil_kalkulasi.get("kebutuhan_harian_liter"),
        "cadangan_air_liter": hasil_kalkulasi.get("cadangan_air_liter"),
        "durasi_hari": hasil_kalkulasi.get("durasi_hari"),
        "status": hasil_kalkulasi.get("status"),
        "alasan_status": hasil_kalkulasi.get("alasan_status"),
    },
    "rekomendasi": rekomendasi_rule_based,
    "waktu_perhitungan_utc": hasil_kalkulasi.get("waktu_perhitungan_utc"),
}

print(json.dumps(ringkasan_output, ensure_ascii=False, indent=2))

{
  "input": {
    "lokasi": "Rongkop, Gunungkidul",
    "jumlah_anggota": 4,
    "sumber_air_utama": "Sumur",
    "kondisi_sumber_air": "Terbatas",
    "metode_volume": "kapasitas × persentase",
    "kebutuhan_mhm": true,
    "jumlah_anggota_mhm": 2
  },
  "hasil_kalkulasi": {
    "kebutuhan_dasar_liter": 60.0,
    "tambahan_air_mhm_liter": 10.0,
    "kebutuhan_harian_liter": 70.0,
    "cadangan_air_liter": 450.0,
    "durasi_hari": 6.43,
    "status": "Waspada",
    "alasan_status": "Cadangan tersimpan diperkirakan cukup lebih dari 3 hari, tetapi sumber air utama sedang terbatas."
  },
  "rekomendasi": {
    "status_sumber": "Waspada",
    "ringkasan_rule_based": "Cadangan atau kondisi sumber air menunjukkan perlunya penghematan dan persiapan sumber alternatif. Kebutuhan MHM yang diinput telah diperhitungkan dalam estimasi kebutuhan harian.",
    "rekomendasi_id": [
      "hemat_air",
      "pantau_cadangan",
      "sumber_alternatif",
      "kebersihan_mhm"
    ],
    "rekomendasi":

## ⚠️ 9. Keterbatasan dan Responsible AI

### Keterbatasan kalkulator dan data
- **Acuan kebutuhan dasar:** Kalkulator menggunakan 15 liter/orang/hari sebagai acuan kebutuhan air minimum dalam konteks darurat. Angka ini bukan kebutuhan ideal untuk semua aktivitas rumah tangga dan tidak selalu mencerminkan kebutuhan aktual setiap wilayah.
- **MHM:** Alokasi 5 liter/orang/hari untuk setiap anggota keluarga yang memerlukan Manajemen Kebersihan Menstruasi (MHM) adalah keputusan operasional AIRA, bukan angka minimum universal yang secara khusus ditetapkan oleh Sphere atau UNICEF. Besaran dan penerapannya perlu dievaluasi berdasarkan panduan yang relevan serta kondisi lokal.
- **Kondisi sumber:** Kategori Normal/Terbatas/Tidak berfungsi/Belum diketahui berasal dari input pengguna, bukan sensor atau API real-time.
- **Ambang status:** Batas 1 dan 3 hari adalah aturan awal prototype, bukan klasifikasi resmi kekeringan.
- **Kualitas air:** Volume dan durasi tidak menunjukkan apakah air aman diminum.
- **Data uji:** Input pada notebook ini adalah skenario simulasi, bukan data rumah tangga nyata. Jangan menyimpan data yang dapat mengidentifikasi pengguna di repository publik tanpa dasar dan izin yang sesuai.
- **Di luar cakupan pipeline ini:** Simulasi panen air hujan (luas atap dan curah hujan) baru ditambahkan setelah metode dan sumber datanya ditetapkan.

### Responsible AI: bagaimana AI dibatasi pada pipeline ini
- **Angka dan status berasal dari kode.** AI tidak menghitung ulang dan tidak boleh mengubah angka atau status.
- **AI hanya memilih dari katalog.** Teks tindakan selalu berasal dari katalog; ID di luar katalog dibuang dan tindakan wajib (misalnya MHM, serta Navigation Air pada Mode Krisis) tetap ditambahkan bila AI melewatkannya.
- **Ada jalur cadangan.** Jika AI gagal atau keluarannya tidak valid, rekomendasi rule-based tetap ditampilkan sebagai referensi utama.
- **Data yang dikirim dibatasi.** Hanya hasil kalkulasi yang dikirim ke AI; lokasi tidak dikirim.
- **Keluaran LLM bisa keliru.** Alasan yang ditulis AI belum diverifikasi otomatis dan perlu ditinjau manusia. Pengujian dengan API asli baru dilakukan pada satu skenario.

## 🚀 10. Next Steps

### Kendala yang ditemukan
- **Model AI sibuk (error 503).** Pada uji awal Gemini menolak permintaan dengan `503 UNAVAILABLE` sehingga pipeline hanya memakai fallback. Perbaikannya: satu fungsi panggilan AI dengan retry bertahap, model cadangan, dan fallback rule-based. Konsekuensinya, waktu respons AI bisa lebih lama saat layanan padat.
- **Nama model dapat berubah.** Nama model dan jadwal penghentiannya ditentukan penyedia, sehingga daftar `GEMINI_MODELS` perlu dipantau dan diperbarui.
- **Pengujian AI masih terbatas.** Output AI dengan API asli baru diuji pada satu skenario, dan kualitas alasan yang ditulis AI belum dinilai secara sistematis.

### Rencana pengembangan berikutnya

**Validasi dan pengujian**
1. Tinjau acuan 15 liter/orang/hari dan ambang status dengan sumber yang relevan serta mentor/tim.
2. Evaluasi alokasi tambahan 5 liter/orang/hari untuk MHM dengan merujuk pada panduan kebersihan menstruasi dan kondisi lokal sebelum digunakan untuk rekomendasi nyata.
3. Uji AI dengan API asli pada skenario Krisis dan Belum Dapat Ditentukan, termasuk skenario tanpa MHM dan dengan beberapa anggota keluarga yang memerlukan MHM.

**Integrasi dengan fitur lain**
4. Hubungkan output kalkulator ke AIRA Assistant dengan mengirim hasil terstruktur (kebutuhan dasar, tambahan MHM, total kebutuhan harian, durasi, status, alasan, rekomendasi). Assistant menjelaskan hasil tanpa menghitung ulang.
5. Integrasikan Navigation Air sebagai aksi lanjutan saat Mode Krisis, dengan catatan titik air harus diverifikasi dan ketersediaannya dikonfirmasi.

**Pengembangan lanjutan**
6. Evaluasi apakah kondisi sumber air perlu menjadi pemicu status atau ditampilkan sebagai indikator terpisah; aturan saat ini adalah keputusan prototype yang harus divalidasi.
7. Lakukan uji pengguna agar label, satuan, dan pesan kesalahan mudah dipahami, serta tambahkan petunjuk memperkirakan volume tangki sesuai desain Figma.

**Pengumpulan**
8. Simpan notebook, README, dan laporan pengujian ke folder repository GitHub kelompok; jangan commit API key atau data pribadi.

## 🏁 Kesimpulan Pipeline Kalkulator Ketahanan Air AIRA

**Apa yang dibangun.** Pipeline awal fitur Kalkulator Ketahanan Air dengan alur: input simulasi rumah tangga → validasi → perhitungan kebutuhan dan durasi cadangan → penetapan status → rekomendasi rule-based → pemilihan tindakan oleh AI (Gemini) dengan validasi dan fallback → output teks. Angka dan status ditentukan kode secara deterministik, sedangkan AI hanya memilih dan mengurutkan tindakan dari katalog yang sudah ditetapkan.

---

### 📊 Hasil pada Skenario Utama

Skenario simulasi: rumah tangga 4 orang (persona di Rongkop, Gunungkidul) memakai sumur dengan kondisi terbatas dan cadangan 450 liter (tangki 1.000 liter terisi 45%). Dua anggota keluarga memerlukan MHM.

- **Kebutuhan dan durasi:** kebutuhan harian 70 liter (60 liter dasar + 10 liter tambahan MHM), sehingga cadangan bertahan sekitar 6,43 hari.
- **Status:** dari durasi saja status-nya Aman, tetapi karena sumber air utama terbatas, aturan sistem menetapkan status akhir **Waspada**.
- **Rekomendasi AI:** Gemini memilih 4 tindakan dengan alasan singkat untuk tiap pilihan. Dibandingkan urutan rule-based, AI menaikkan tindakan kebersihan menstruasi ke prioritas kedua karena ada anggota keluarga yang memerlukannya. Teks tiap tindakan tetap berasal dari katalog dan status tetap Waspada.

### 🧪 Hasil Pengujian

- **11 pengujian status dan validasi lulus**, mencakup batas 1 hari, 3 hari, dan 3,01 hari, kondisi sumber air, input tidak valid, dan perhitungan MHM.
- **10 pengujian lapisan AI lulus** tanpa memanggil API: retry, pindah model cadangan, pembuangan ID di luar katalog, tindakan wajib, serta fallback saat AI error atau keluarannya bukan JSON.
- Kegagalan AI yang sempat terjadi (error 503) dijelaskan pada Bagian 10. Setelah retry, AI berhasil menjawab pada percobaan ke-4, dan model cadangan terbukti dipakai pada uji koneksi.

---

### ✅ Pemenuhan Kriteria Assignment

| Kriteria | Bukti pada notebook |
|---|---|
| Fitur final project dipilih dengan jelas | Kalkulator Ketahanan Air dan rekomendasi tindakan (Bagian 1) |
| Pipeline dapat dijalankan, atau langkah yang belum berhasil dijelaskan | Seluruh sel berjalan; kegagalan AI (503) dijelaskan dan ditangani (Bagian 6 dan 10) |
| Ada output atau bukti pengujian awal | Hasil kalkulasi, rekomendasi rule-based dan AI (Bagian 4–6), serta pengujian (Bagian 7) |
| Hasil, keterbatasan, dan langkah berikutnya dapat dijelaskan | Interpretasi (Bagian 8), keterbatasan dan Responsible AI (Bagian 9), kendala dan rencana (Bagian 10) |
| Kode dan dokumentasi tersedia di repository | Disiapkan untuk diunggah ke folder kelompok pada repository GitHub |

> Pipeline ini belum merupakan kalkulator produksi. Acuan kebutuhan air, alokasi tambahan MHM, ambang status, dan rekomendasi perlu ditinjau sebelum digunakan untuk keputusan nyata.